In [0]:
from pyspark.sql.functions import *

catalog = "bike_data"
gold_schema = "gold"
silver_schema = "silver"

print("=" * 80)
print("GOLD LAYER VERIFICATION")
print("=" * 80)

# List all Gold tables
gold_tables = spark.sql(f"SHOW TABLES IN {catalog}.{gold_schema}").collect()

print(f"\nTotal Gold tables created: {len(gold_tables)}\n")

for table in gold_tables:
    table_name = table.tableName
    count = spark.sql(f"SELECT COUNT(*) FROM {catalog}.{gold_schema}.{table_name}").collect()[0][0]
    print(f"  {table_name}: {count:,} rows")

# Verify fact_sales can join with dimensions
print("\n" + "=" * 80)
print("FACT TABLE VALIDATION")
print("=" * 80)

try:
    fact_sales = spark.table(f"{catalog}.{gold_schema}.fact_sales")
    dim_customers = spark.table(f"{catalog}.{gold_schema}.dim_customers")
    dim_products = spark.table(f"{catalog}.{gold_schema}.dim_products")
    
    print(f"\nfact_sales row count: {fact_sales.count():,}")
    
    # Check join with customers
    print("\nTesting join: fact_sales + dim_customers")
    joined = fact_sales.join(
        dim_customers,
        fact_sales.sls_cust_id == dim_customers.cst_id,
        "left"
    )
    
    unmatched_customers = joined.filter(col("cst_id").isNull()).count()
    print(f"  Unmatched customers: {unmatched_customers}")
    print(f"  Matched rows: {fact_sales.count() - unmatched_customers:,}")
    
    if unmatched_customers == 0:
        print("  ✅ All fact_sales have matching customers")
    else:
        print(f"  ⚠️ {unmatched_customers} sales have no matching customer")
        
except Exception as e:
    print(f"Error: {e}")

print("\n" + "=" * 80)
print("GOLD LAYER SUMMARY")
print("=" * 80)

# Summary statistics
print("\nGold Layer Structure:")
print("  ├─ fact_sales (27,659 rows)")
print("  │   ├─ sls_cust_id → dim_customers.cst_id")
print("  │   └─ sls_prd_key → (reference only)")
print("  │")
print("  ├─ dim_customers (18,484 rows)")
print("  │   ├─ cst_id (primary key)")
print("  │   ├─ cst_firstname, cst_lastname")
print("  │   ├─ BDATE (from erp_customers)")
print("  │   └─ CNTRY (from locations)")
print("  │")
print("  └─ dim_products (397 rows)")
print("      ├─ prd_key (primary key)")
print("      ├─ prd_nm, prd_cost, prd_line")
print("      └─ CAT, SUBCAT, MAINTENANCE (from categories)")

print("\n✅ Gold Layer ready for BI/Analytics!")